# SparseGuard-NIDS Full Project Control

Run stages sequentially in Colab. This notebook mounts Drive, locates the implementation and datasets, then executes the pipeline.


In [ ]:
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception as e:
    print('No previous Drive mount to flush:', e)
drive.mount('/users/', force_remount=True, timeout_ms=600000)
print('Drive mounted')


In [ ]:
from google.colab import drive
try:
    drive.mount('/users/', force_remount=False, timeout_ms=600000)
except Exception as e:
    print('Drive mount status:', e)

import os, sys, time, subprocess, importlib
from pathlib import Path
try:
    import shap
    print('SHAP available', shap.__version__)
except Exception:
    print('Installing SHAP for DeepSHAP/SHAP XAI...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=False)

import os, sys
from pathlib import Path

def find_sparseguard_project_root():
    env_root = os.environ.get("SPARSEGUARD_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
        Path.cwd(),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("IMPLEMENTATION/src/sparseguard_pipeline.py"))
    valid = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        if (candidate / "src" / "sparseguard_pipeline.py").exists():
            score = 0
            score += 10 if "LocalDrive1/OnlyScholar/Projects" in str(candidate) else 0
            score += 5 if (candidate / "EXPERIMENT" / "results" / "sparseguard_best.pt").exists() else 0
            score += 3 if (candidate / "Q1_VALIDATION" / "scripts" / "q1_sci_validation.py").exists() else 0
            valid.append((score, candidate))
    if not valid:
        raise FileNotFoundError("Could not locate finalized IMPLEMENTATION/src/sparseguard_pipeline.py. Mount Drive or set SPARSEGUARD_ROOT.")
    return sorted(valid, key=lambda item: (item[0], len(str(item[1]))), reverse=True)[0][1]

def find_dataset_root():
    env_root = os.environ.get("SPARSEGUARD_DATASET_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("X-IIoTID/X-IIoTID dataset.csv"))
    for candidate in candidates:
        if (candidate / "X-IIoTID" / "X-IIoTID dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate X-IIoTID dataset root. Mount Drive or set SPARSEGUARD_DATASET_ROOT.")

PROJECT_ROOT = find_sparseguard_project_root()
DATASET_ROOT = find_dataset_root()
os.environ["SPARSEGUARD_ROOT"] = str(PROJECT_ROOT)
os.environ["SPARSEGUARD_DATASET_ROOT"] = str(DATASET_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATASET_ROOT", DATASET_ROOT)

import sparseguard_pipeline as sg
sg = importlib.reload(sg)
print('Running remaining sections from', sg.PROJECT_ROOT)
print('Dataset root', sg.DATASET_ROOT)

start = time.perf_counter()
xai = sg.compute_xai_attributions(sample_n=1024, background_n=32, run_deepshap=False)
print('XAI complete', xai)

attacks = sg.run_sparse_adversarial_attacks(sample_n=2048, steps=8, epsilons=[0.01, 0.03, 0.05, 0.10], topk_values=[3, 5, 10, 20])
print('Sparse adversarial attacks complete', attacks[:3], 'rows', len(attacks))

vae = sg.train_vae_reconstruction_detector(max_epochs=15, batch_size=2048, sample_n_train=60000, sample_n_eval=10000)
print('VAE reconstruction detector complete', vae)

profile = sg.profile_sparseguard_runtime(sample_n=4096, repeats=10)
print('Profiling complete', profile)

print('Remaining Q1 sections complete in seconds', round(time.perf_counter() - start, 3))


In [ ]:
import importlib, time
import sparseguard_pipeline as sg
sg = importlib.reload(sg)
print('Recovery run source', sg.PROJECT_ROOT)
start = time.perf_counter()

xai = sg.compute_xai_attributions(sample_n=1024, background_n=32, run_deepshap=False)
print('XAI summary complete', xai)

attacks = sg.run_sparse_adversarial_attacks(sample_n=2048, steps=8, epsilons=[0.01, 0.03, 0.05, 0.10], topk_values=[3, 5, 10, 20])
print('Sparse adversarial attacks complete rows', len(attacks))

vae = sg.train_vae_reconstruction_detector(max_epochs=15, batch_size=2048, sample_n_train=60000, sample_n_eval=10000)
print('VAE reconstruction detector complete', vae)

profile = sg.profile_sparseguard_runtime(sample_n=4096, repeats=10)
print('Profiling complete', profile)
print('Remaining Q1 recovery sections complete in seconds', round(time.perf_counter() - start, 3))


Recovery run source /users/
XAI summary complete {'sample_n': 1024, 'background_n': 32, 'seconds': 1.801, 'top10_gradient_input_features': [{'feature': 'total_bytes', 'semantic_group': 'packet_volume', 'mean_abs_gradient_input': 0.0017781684873625636, 'mean_signed_gradient_input': -0.00018148269737139344, 'std_abs_gradient_input': 0.012562751770019531}, {'feature': 'Service', 'semantic_group': 'protocol_semantics', 'mean_abs_gradient_input': 0.0013203609269112349, 'mean_signed_gradient_input': 0.000466244004201144, 'std_abs_gradient_input': 0.0022800914011895657}, {'feature': 'Protocol', 'semantic_group': 'protocol_semantics', 'mean_abs_gradient_input': 0.0006570271798409522, 'mean_signed_gradient_input': -8.613472164142877e-05, 'std_abs_gradient_input': 0.000905472319573164}, {'feature': 'Duration', 'semantic_group': 'flow_timing', 'mean_abs_gradient_input': 0.0005913615459576249, 'mean_signed_gradient_input': -0.00015575072029605508, 'std_abs_gradient_input': 0.001206144574098289}, {

In [ ]:
import importlib, time
import sparseguard_pipeline as sg
sg = importlib.reload(sg)
start = time.perf_counter()
ks = sg.compute_kernel_shap_attributions(sample_n=96, background_n=24, nsamples=128)
print('KernelSHAP complete', ks)
print('KernelSHAP seconds total', round(time.perf_counter() - start, 3))


  0%|          | 0/96 [00:00<?, ?it/s]

KernelSHAP complete {'method': 'KernelSHAP', 'sample_n': 96, 'background_n': 24, 'nsamples': 128, 'seconds': 16.594, 'top10_kernel_shap_features': [{'feature': 'read_write_physical.process', 'semantic_group': 'unknown_numeric', 'mean_abs_kernel_shap': 0.12637316620961372, 'mean_signed_kernel_shap': -0.07025358930167525}, {'feature': 'Service', 'semantic_group': 'protocol_semantics', 'mean_abs_kernel_shap': 0.11725834408737855, 'mean_signed_kernel_shap': 0.02052876422396471}, {'feature': 'Protocol', 'semantic_group': 'protocol_semantics', 'mean_abs_kernel_shap': 0.04699842212393308, 'mean_signed_kernel_shap': 0.010556276137497685}, {'feature': 'is_syn_only', 'semantic_group': 'unknown_numeric', 'mean_abs_kernel_shap': 0.026281416464476844, 'mean_signed_kernel_shap': -0.003628620273522903}, {'feature': 'Is_SYN_ACK', 'semantic_group': 'unknown_numeric', 'mean_abs_kernel_shap': 0.01682335832475314, 'mean_signed_kernel_shap': 0.00650006889400349}, {'feature': 'Des_port', 'semantic_group': '

In [ ]:
from google.colab import drive
try:
    drive.mount('/users/', force_remount=False, timeout_ms=600000)
except Exception as e:
    print('Drive mount status:', e)

import os, sys, runpy, time
from pathlib import Path

def find_sparseguard_project_root():
    candidates = [
        Path(os.environ.get('SPARSEGUARD_ROOT', '')),
        Path('/users/'),
        Path('/users/'),
    ]
    drive_root = Path('/users/')
    if drive_root.exists():
        candidates.extend(p.parents[2] for p in drive_root.rglob('IMPLEMENTATION/ROBUST_XAI_FRAMEWORK/scripts/robust_xai_framework.py'))
    valid = []
    for candidate in candidates:
        if str(candidate) and (candidate / 'ROBUST_XAI_FRAMEWORK' / 'scripts' / 'robust_xai_framework.py').exists():
            score = 0
            score += 10 if 'LocalDrive1/OnlyScholar/Projects' in str(candidate) else 0
            score += 5 if (candidate / 'src' / 'sparseguard_pipeline.py').exists() else 0
            score += 3 if (candidate / 'XAI' / 'results' / 'kernel_shap_feature_importance.csv').exists() else 0
            valid.append((score, candidate))
    if not valid:
        raise FileNotFoundError('Could not locate finalized Robust-XAI script.')
    return sorted(valid, key=lambda item: (item[0], len(str(item[1]))), reverse=True)[0][1]

def find_dataset_root():
    candidates = [Path(os.environ.get('SPARSEGUARD_DATASET_ROOT', '')), Path('/users/')]
    drive_root = Path('/users/')
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob('X-IIoTID/X-IIoTID dataset.csv'))
    for candidate in candidates:
        if str(candidate) and (candidate / 'X-IIoTID' / 'X-IIoTID dataset.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate X-IIoTID dataset root.')

PROJECT_ROOT = find_sparseguard_project_root()
DATASET_ROOT = find_dataset_root()
os.environ['SPARSEGUARD_ROOT'] = str(PROJECT_ROOT)
os.environ['SPARSEGUARD_DATASET_ROOT'] = str(DATASET_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
script = PROJECT_ROOT / 'ROBUST_XAI_FRAMEWORK' / 'scripts' / 'robust_xai_framework.py'
print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Running:', script)
t0 = time.perf_counter()
runpy.run_path(str(script), run_name='__main__')
print('Robust-XAI extension finished in seconds', round(time.perf_counter() - t0, 3))


Mounted at /users/
Project root: /users/
Dataset root: /users/
Running: /users/
ROBUST_XAI_FRAMEWORK complete /users/
Robust-XAI extension finished in seconds 18.948


In [ ]:
from google.colab import drive
try:
    drive.mount('/users/', force_remount=False, timeout_ms=600000)
except Exception as e:
    print('Drive mount status:', e)
from pathlib import Path
import base64, os
PROJECT_ROOT = Path(os.environ.get('SPARSEGUARD_ROOT', '/users/'))
if not (PROJECT_ROOT / 'src' / 'sparseguard_pipeline.py').exists():
    matches = list(Path('/users/').rglob('IMPLEMENTATION/src/sparseguard_pipeline.py'))
    if matches:
        PROJECT_ROOT = matches[0].parents[1]
script_dir = PROJECT_ROOT / 'Q1_VALIDATION' / 'scripts'
script_dir.mkdir(parents=True, exist_ok=True)
script_path = script_dir / 'q1_sci_validation.py'
script_path.write_text(base64.b64decode('...').decode('utf-8'), encoding='utf-8')
for folder in ['results', 'DOCUMANTATION', 'notebooks']:
    (PROJECT_ROOT / 'Q1_VALIDATION' / folder).mkdir(parents=True, exist_ok=True)
print('Wrote script:', script_path, script_path.exists(), script_path.stat().st_size)
print('Q1 entries:', sorted(str(p.relative_to(PROJECT_ROOT / 'Q1_VALIDATION')) for p in (PROJECT_ROOT / 'Q1_VALIDATION').rglob('*'))[:20])

Drive already mounted at /users/; to attempt to forcibly remount, call drive.mount("/users/", force_remount=True).
Wrote script: /users/ True 26257
Q1 entries: ['DOCUMANTATION', 'notebooks', 'notebooks/01_Q1_SCI_VALIDATION_COLAB.ipynb', 'results', 'scripts', 'scripts/q1_sci_validation.py']


In [ ]:
from google.colab import drive
try:
    drive.mount('/users/', force_remount=False, timeout_ms=600000)
except Exception as e:
    print('Drive mount status:', e)

import os, sys, runpy, time
from pathlib import Path

def find_sparseguard_project_root():
    candidates = [
        Path(os.environ.get('SPARSEGUARD_ROOT', '')),
        Path('/users/'),
        Path('/users/'),
    ]
    drive_root = Path('/users/')
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob('IMPLEMENTATION/src/sparseguard_pipeline.py'))
    valid = []
    for candidate in candidates:
        if str(candidate) and (candidate / 'src' / 'sparseguard_pipeline.py').exists():
            score = 0
            score += 10 if 'LocalDrive1/OnlyScholar/Projects' in str(candidate) else 0
            score += 5 if (candidate / 'EXPERIMENT' / 'results' / 'sparseguard_best.pt').exists() else 0
            score += 3 if (candidate / 'Q1_VALIDATION' / 'scripts' / 'q1_sci_validation.py').exists() else 0
            valid.append((score, candidate))
    if not valid:
        raise FileNotFoundError('Could not locate finalized SparseGuard IMPLEMENTATION root.')
    return sorted(valid, key=lambda item: (item[0], len(str(item[1]))), reverse=True)[0][1]

def find_dataset_root():
    candidates = [
        Path(os.environ.get('SPARSEGUARD_DATASET_ROOT', '')),
        Path('/users/'),
        Path('/users/'),
    ]
    drive_root = Path('/users/')
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob('X-IIoTID/X-IIoTID dataset.csv'))
    for candidate in candidates:
        if str(candidate) and (candidate / 'X-IIoTID' / 'X-IIoTID dataset.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate X-IIoTID dataset root.')

PROJECT_ROOT = find_sparseguard_project_root()
DATASET_ROOT = find_dataset_root()
os.environ['SPARSEGUARD_ROOT'] = str(PROJECT_ROOT)
os.environ['SPARSEGUARD_DATASET_ROOT'] = str(DATASET_ROOT)
os.environ['Q1_MAIN_SAMPLE_N'] = '60000'
os.environ['Q1_BOOTSTRAPS'] = '500'
os.environ['Q1_DOMAIN_MAX_ROWS'] = '90000'
os.environ['Q1_DOMAIN_SEEDS'] = '11,22,33,44,55'
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

script = PROJECT_ROOT / 'Q1_VALIDATION' / 'scripts' / 'q1_sci_validation.py'
if not script.exists():
    raise FileNotFoundError(f'Missing Q1 validation script in finalized root: {script}')
if not (PROJECT_ROOT / 'EXPERIMENT' / 'results' / 'sparseguard_best.pt').exists():
    raise FileNotFoundError('Missing finalized model checkpoint; run EXPERIMENT first.')

print('Project root:', PROJECT_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Running:', script)
t0 = time.perf_counter()
runpy.run_path(str(script), run_name='__main__')
print('Q1 SCI validation finished in seconds', round(time.perf_counter() - t0, 3))


Script visible: /users/ True 26257
Running Q1 validation now
Q1_VALIDATION complete /users/
Q1 methodology package ready: True
Q1 SCI validation finished in seconds 940.2


In [6]:
from pathlib import Path
import base64, json, importlib.util, os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
PROJECT_ROOT = Path(os.environ.get('SPARSEGUARD_ROOT', '/users/'))
if not (PROJECT_ROOT / 'src' / 'sparseguard_pipeline.py').exists():
    matches = list(Path('/users/').rglob('IMPLEMENTATION/src/sparseguard_pipeline.py'))
    if matches:
        PROJECT_ROOT = matches[0].parents[1]
section = PROJECT_ROOT / 'Q1_VALIDATION'
script_path = section / 'scripts' / 'q1_sci_validation.py'
script_path.write_text(base64.b64decode('...').decode('utf-8'), encoding='utf-8')
spec = importlib.util.spec_from_file_location('q1_sci_validation_repair', script_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
results = section / 'results'
pred = pd.read_csv(results / 'main_model_validation_predictions.csv')
ece, mce, bins = mod.expected_calibration_error(pred['target'].values, pred['p_attack'].values, bins=15)
bins.to_csv(results / 'calibration_reliability_bins.csv', index=False)
plt.figure(figsize=(6, 5))
plot_bins = bins[bins['count'] > 0]
plt.plot([0, 1], [0, 1], '--', color='black', linewidth=1)
sns.scatterplot(data=plot_bins, x='mean_predicted_attack_probability', y='empirical_attack_rate', size='count', legend=False)
plt.xlim(0, 1); plt.ylim(0, 1); plt.tight_layout()
plt.savefig(results / 'calibration_reliability_diagram.png', dpi=220)
plt.close()
cal_path = results / 'calibration_and_statistical_reliability_summary.json'
cal = json.loads(cal_path.read_text(encoding='utf-8'))
cal['ece'] = ece
cal['mce'] = mce
cal['calibration_definition'] = 'Binary positive-class calibration: weighted gap between empirical attack rate and mean predicted attack probability per probability bin.'
cal_path.write_text(json.dumps(cal, indent=2, sort_keys=True), encoding='utf-8')
ready_path = results / 'q1_sci_readiness_summary.json'
ready = json.loads(ready_path.read_text(encoding='utf-8'))
ready['run_summary']['calibration']['ece'] = ece
ready['run_summary']['calibration']['mce'] = mce
ready['run_summary']['calibration']['calibration_definition'] = cal['calibration_definition']
ready_path.write_text(json.dumps(ready, indent=2, sort_keys=True), encoding='utf-8')
print('Calibration repaired. ECE=', round(ece, 6), 'MCE=', round(mce, 6))
print('Script patched:', script_path, script_path.stat().st_size)


Calibration repaired. ECE= 0.001753 MCE= 0.200837
Script patched: /users/ 26354


In [ ]:
import os, sys
from pathlib import Path

def find_sparseguard_project_root():
    env_root = os.environ.get("SPARSEGUARD_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
        Path.cwd(),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("IMPLEMENTATION/src/sparseguard_pipeline.py"))
    valid = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        if (candidate / "src" / "sparseguard_pipeline.py").exists():
            score = 0
            score += 10 if "LocalDrive1/OnlyScholar/Projects" in str(candidate) else 0
            score += 5 if (candidate / "EXPERIMENT" / "results" / "sparseguard_best.pt").exists() else 0
            score += 3 if (candidate / "Q1_VALIDATION" / "scripts" / "q1_sci_validation.py").exists() else 0
            valid.append((score, candidate))
    if not valid:
        raise FileNotFoundError("Could not locate finalized IMPLEMENTATION/src/sparseguard_pipeline.py. Mount Drive or set SPARSEGUARD_ROOT.")
    return sorted(valid, key=lambda item: (item[0], len(str(item[1]))), reverse=True)[0][1]

def find_dataset_root():
    env_root = os.environ.get("SPARSEGUARD_DATASET_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([
        Path("/users/"),
        Path("/users/"),
    ])
    drive_root = Path("/users/")
    if drive_root.exists():
        candidates.extend(p.parents[1] for p in drive_root.rglob("X-IIoTID/X-IIoTID dataset.csv"))
    for candidate in candidates:
        if (candidate / "X-IIoTID" / "X-IIoTID dataset.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate X-IIoTID dataset root. Mount Drive or set SPARSEGUARD_DATASET_ROOT.")

PROJECT_ROOT = find_sparseguard_project_root()
DATASET_ROOT = find_dataset_root()
os.environ["SPARSEGUARD_ROOT"] = str(PROJECT_ROOT)
os.environ["SPARSEGUARD_DATASET_ROOT"] = str(DATASET_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATASET_ROOT", DATASET_ROOT)


In [ ]:
import importlib
import sparseguard_pipeline as sg
sg = importlib.reload(sg)
print('Pipeline loaded/reloaded')
print('Project', sg.PROJECT_ROOT)
print('Datasets', sg.DATASET_ROOT)


## Stage 1: EDA


In [ ]:
sg.eda_x_iiotid()
print("EDA complete")


## Stage 2: Preprocessing


In [ ]:
sg.preprocess_x_iiotid()
print("Preprocessing complete")


## Stage 3: Main Experiment


In [ ]:
sg.train_sparseguard()
print("Training complete")


## Stage 4-7: Test Evaluation, Ablation, Robustness, External Plan, Paper Assets


In [ ]:
import importlib
import sparseguard_pipeline as sg
sg = importlib.reload(sg)
test_metrics = sg.evaluate_sparseguard_test()
print('Test evaluation complete', test_metrics)
ablation_rows = sg.run_ablation_experiments()
print('Ablation complete', ablation_rows)
robustness_rows = sg.run_robustness_actual()
print('Robustness complete', robustness_rows)
external_rows = sg.run_cross_dataset_validation()
print('External validation complete', external_rows)
sg.write_paper_asset_plan()
print('Evaluation and paper assets complete')


Test evaluation complete {'accuracy': 0.9968745394704525, 'balanced_accuracy': 0.9968415084357354, 'precision': 0.9979372832881982, 'recall': 0.9956315889148413, 'f1': 0.9967831027574308, 'mcc': 0.9937465461691144, 'brier': 0.0025629194634204143, 'confusion_matrix': [[83488, 163], [346, 78859]], 'roc_auc': 0.9998646884734064, 'average_precision': 0.9998712202253786}
Ablation complete [{'variant': 'full_semantic_multipath_repeat', 'best_val_f1': 0.9968138370505234, 'test_accuracy': 0.9965552389841332, 'test_balanced_accuracy': 0.9965125752952201, 'test_precision': 0.9979611478357774, 'test_recall': 0.994949813774383, 'test_f1': 0.9964532057077466, 'test_mcc': 0.9931090712839132, 'test_brier': 0.00274605582577915, 'test_roc_auc': 0.9998538980177086, 'test_average_precision': 0.9998623100838139}, {'variant': 'no_reconstruction_loss', 'best_val_f1': 0.9966886580217892, 'test_accuracy': 0.9966105025298423, 'test_balanced_accuracy': 0.9965787843921599, 'test_precision': 0.997608533360327, 't

In [ ]:
import importlib, time
import sparseguard_pipeline as sg
sg = importlib.reload(sg)
start = time.perf_counter()
external_rows = sg.run_cross_dataset_validation(max_rows_per_dataset=240000)
print('External validation complete in seconds', round(time.perf_counter() - start, 3))
for row in external_rows:
    print(row)
sg.write_paper_asset_plan()
print('Paper asset plan refreshed')


External validation complete in seconds 203.652
{'protocol': 'within_dataset', 'train_dataset': 'X-IIoTID', 'test_dataset': 'X-IIoTID', 'accuracy': 0.9949305555555555, 'balanced_accuracy': 0.9949305555555555, 'precision': 0.9956327018832235, 'recall': 0.9942222222222222, 'f1': 0.9949269621537478, 'mcc': 0.9898621044107351, 'brier': 0.004043649971255142, 'roc_auc': 0.9997979363425925, 'average_precision': 0.9998068625011269, 'train_seconds': 2.769203133000701, 'inference_seconds': 0.23246405200006848, 'iterations': 180, 'train_rows': 168000, 'test_rows': 72000}
{'protocol': 'within_dataset', 'train_dataset': 'CIC-IIoT-2025', 'test_dataset': 'CIC-IIoT-2025', 'accuracy': 0.97875, 'balanced_accuracy': 0.97875, 'precision': 0.9940377228687726, 'recall': 0.9632777777777778, 'f1': 0.9784160483029088, 'mcc': 0.9579587606938489, 'brier': 0.017270195403356014, 'roc_auc': 0.9961301786265433, 'average_precision': 0.9957345547972127, 'train_seconds': 2.8038369890000467, 'inference_seconds': 0.22085